# Six things agents are actually used for

Each section is a different shape. Run them in order
and watch ; nothing here needs writing.

| # | Job | Shape |
| --- | --- | --- |
| 1 | Pull fields out of a messy report | structured output, no tools |
| 2 | Answer a question about records | one tool, then reasoning |
| 3 | Compare two things | a loop, several tool calls |
| 4 | Check something against a policy | judgement, with a hard rule around it |
| 5 | Triage, and know when to give up | abstention |
| 6 | Reformat some dates | no agent at all |

The last one matters most. Knowing when *not* to reach for an agent is the
part that saves money.

In [2]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

model=qwen3.8-flash


## 1. Getting fields out of free text

Somebody types an incident report in whatever words they like. We need the
same four fields every time, so the next system can read them.

No tools here. The model is asked for a shape and returns that shape.

In [3]:
from pydantic import BaseModel, Field
from typing import Literal

REPORT = ("Badge scanners at the Rotterdam depot stopped reading cards at "
          "06:00 this morning. Around 40 drivers cannot get onto the loading "
          "bay. The night shift was not affected. Reported by W. de Vries.")


class Incident(BaseModel):
    site: str = Field(description="where it happened")
    severity: Literal["low", "medium", "high"]
    people_affected: int
    reported_by: str


llm.with_structured_output(Incident).invoke(REPORT)

Incident(site='Rotterdam depot', severity='high', people_affected=40, reported_by='W. de Vries')

The result is a typed object. It can be passed to the next step ; loose text has to be read by a
person first. Rerun it with the report reworded ; the fields stay
in the same places.

## 2. Answering a question about records

The data is in a table. Give the agent a way to read it and let it work out
which rows matter.

In [4]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

ORDERS = [
    {"id": "A-1", "customer": "Nova Logistics", "status": "delivered", "eur": 4200},
    {"id": "A-2", "customer": "Nova Logistics", "status": "delayed",   "eur": 1150},
    {"id": "A-3", "customer": "Helix Medical",  "status": "delivered", "eur": 980},
    {"id": "A-4", "customer": "Bakker & Zoon",  "status": "delayed",   "eur": 300},
]


@tool
def find_orders(status: str) -> list:
    """Find orders by status. Use 'delayed' or 'delivered'."""
    return [o for o in ORDERS if o["status"] == status]


agent = create_agent(model=llm, tools=[find_orders],
                     system_prompt="Answer from the tool only. Be brief.")

reply = agent.invoke({"messages": [HumanMessage(
    "Which customers have delayed orders, and what are they worth in total?")]})
print(reply["messages"][-1].content)

- **Nova Logistics** — €1,150
- **Bakker & Zoon** — €300

**Total: €1,450**


It had to fetch the rows, then add them up. Neither step was written by
us : the filtering is the tool, the arithmetic and the wording are the model.

## 3. Comparing two things

One question produces several tool calls. The second lookup was not asked for ;
the model decided it needed one.

In [5]:
PLANS = {
    "Standard":   {"response": "8 hours", "on_site": "next day", "eur_month": 250},
    "Enterprise": {"response": "1 hour",  "on_site": "same day", "eur_month": 900},
}

CALLS = []


@tool
def get_plan(name: str) -> dict:
    """Look up what a support plan includes. Use 'Standard' or 'Enterprise'."""
    CALLS.append(name)
    return PLANS[name]


agent = create_agent(model=llm, tools=[get_plan],
                     system_prompt="Compare using the tool. Two sentences maximum.")

reply = agent.invoke({"messages": [HumanMessage(
    "We are on Standard. Is Enterprise worth the extra money if our depots "
    "cannot afford to be down for a working day?")]})

print(reply["messages"][-1].content)
print("\ntool was called for:", CALLS)

Yes — Enterprise cuts response time from 8 hours to 1 hour and upgrades on-site service from next-day to same-day, which directly addresses your inability to tolerate a full working day of downtime. The extra €650/month buys you roughly a full day of avoided outage per incident, so it's worth it if even one breakdown would cost more than that in lost operations.

tool was called for: ['Standard', 'Enterprise']


Two calls, then an answer that weighs them. This is the loop doing its
job : the model decided it needed a second fact before it could reply.

## 4. Checking something against a policy

Judgement the model is good at, wrapped in a rule it cannot argue with. The
hard check runs first and the model never sees a blocked message.

In [6]:
import re

POLICY = """A message must be escalated to a human when it:
- asks for money back, a refund, or a credit
- threatens legal action or mentions a regulator
- concerns a data breach or lost personal data
Otherwise the service desk may answer it."""

IBAN = re.compile(r"\b[A-Z]{2}\d{2}[A-Z0-9]{10,}\b")

MESSAGES = [
    "The scanner in bay 3 is offline again, third time this week.",
    "We want the March overcharge refunded or we will raise it with the AFM.",
    "Please send the refund to NL91ABNA0417164300, that is our account.",
]


def check(text):
    if IBAN.search(text):
        return "BLOCKED before the model : contains a bank account number"
    verdict = llm.invoke(
        f"{POLICY}\n\nMessage: {text}\n\n"
        "Reply with ESCALATE or ANSWER, then five words of reason.").content
    return verdict.strip()


for m in MESSAGES:
    print(f"{m[:52]:54} -> {check(m)}")

The scanner in bay 3 is offline again, third time th   -> ANSWER Technical issue no refund legal


We want the March overcharge refunded or we will rai   -> ESCALATE refund request legal threat regulator
Please send the refund to NL91ABNA0417164300, that i   -> BLOCKED before the model : contains a bank account number


The third message never reaches the model. A regular expression is not clever, which is why it is
a reasonable check to put in front of a model.

## 5. Knowing when to give up

An agent that always answers is worse than one that sometimes says it cannot.
Here it is given a way out, and a ticket it has no hope of classifying.

In [7]:
QUEUES = "access, hardware, billing, unknown"

TICKETS = [
    "I cannot log in, the reset mail never arrives.",
    "The card reader at gate 2 is dead.",
    "As discussed on Tuesday, please proceed as agreed.",
]


def route(text):
    return llm.invoke(
        f"Route this ticket to one of: {QUEUES}. Answer with one word. "
        f"Use 'unknown' if the ticket does not say enough.\n\n{text}"
    ).content.strip().lower()


for t in TICKETS:
    print(f"{route(t):10} <- {t[:56]}")

access     <- I cannot log in, the reset mail never arrives.


hardware   <- The card reader at gate 2 is dead.


unknown    <- As discussed on Tuesday, please proceed as agreed.


The third has no answer in it, and `unknown` is the correct one. Take the
word `unknown` out of that prompt and the model will pick something anyway ;
that is how confident wrong answers get made.

## 6. The one that should not be an agent

Dates arriving in four formats, needing one. It is tempting to ask a model.

In [8]:
from datetime import datetime

RAW = ["2026-03-01", "01/03/2026", "1 March 2026", "2026.03.01"]
FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%d %B %Y", "%Y.%m.%d"]


def parse(value):
    for f in FORMATS:
        try:
            return datetime.strptime(value, f).date()
        except ValueError:
            pass


print([str(parse(v)) for v in RAW])

['2026-03-01', '2026-03-01', '2026-03-01', '2026-03-01']


Instant, free, and identical every time it runs. A model would cost a
call per row, take a second each, and occasionally invent a date.

The rule that follows from this : **if the rules are known and fixed, write
them.** Reach for an agent when the steps depend on something we cannot know
until we look, which is what every other section on this page has in common.

## Try one of these

- Section 2 : ask "which customer is worth most to us?" ; the tool only filters
  by status, so watch what it does with a question the tool cannot answer
- Section 4 : add "mentions a competitor" to the policy and re-run
- Section 5 : delete `Use 'unknown' if...` and see the third ticket get a
  confident wrong queue
- Section 1 : reword the report completely and check the fields still land